In [ ]:
import os
import random
import numpy as np
from PIL import Image
from tqdm import tqdm
from pathlib import Path
from collections import Counter
from matplotlib import pyplot as plt

import torchvision.transforms as transforms
from torch.utils.data import random_split, DataLoader
from torch import Generator

import torch
from transformers import CLIPProcessor

# Dataset image sizes

Datasets

- Real faces: `ffhq_real_faces`
    - 3143 images
    - these are all in `png` format
- Diffusion-generated faces (set 1): `AIS-4SD/StableDiffusion-3-faces-20250203-1545`
    - 500 images
    - these are all in `png` format
- Diffusion-generated faces (set 2): `SFHQ-T2I`
    - 1724 images
    - these are all in `jpg` format

Note that I have also combined all of the diffusion-generated images into a single folder: "all_diffusion_images" to make the de-duplication step easier.

For each of these datasets, we are interested in the image size and whether it's consistent for all images within the dataset.

In [ ]:
def get_image_sizes(image_folder_path: Path) -> list[tuple[int, int]]:
    """
    Get the sizes of all images in a given folder.
    """
    image_names = os.listdir(image_folder_path)
    image_sizes = []
    for i in tqdm(range(len(image_names))):
        image_name = image_names[i]
        if image_name != ".DS_Store":
            test_image_path = image_folder_path / image_name
            test_img = Image.open(test_image_path)
            image_sizes.append(test_img.size)
    return image_sizes

## Real images

In [ ]:
real_images_path = Path("data/ffhq_real_faces")
image_sizes = get_image_sizes(real_images_path)
Counter(image_sizes)

All images in this dataset have size (1024, 1024)

## Synthetic images

### AIS-4SD

In [ ]:
synth_images_1_path = Path("data/AIS-4SD/StableDiffusion-3-faces-20250203-1545")
image_sizes = get_image_sizes(synth_images_1_path)
Counter(image_sizes)

All images in this dataset have size (768, 768)

### SFHQ-T2I

In [ ]:
synth_images_2_path = Path("data/SFHQ-T2I")
image_sizes = get_image_sizes(synth_images_2_path)
Counter(image_sizes)

All images in this dataset have size (1024, 1024)

## Summary

- All 3143 real images have size (1024, 1024)
- 1724 of the diffusion-generated images have size (1024, 1024), but 500 of them have size (768, 768)

I could upscale the smaller images, but it would be safer (less likely to introduce image artifacts) to reduce the size of the larger images to (768, 768).

# Check for duplicates

There are several reasons why we don't want any duplicates in our dataset:

- If the same image appears in the train and test datasets, then we are not truly testing the model since it has already seen the image
- If one class contains duplicates but the other doesn't, then we have a class imbalance

We could identify duplicates by hashing the images, however I came across an open-source library called `fiftyone` which enables dataset visualisation and tools such as identifying duplicates. This seems like a good opportunity to try it out!

In [ ]:
import fiftyone as fo
import fiftyone.brain as fob

In [ ]:
# load a folder of images as a fiftyone dataset
dataset = fo.Dataset.from_images_dir("data/all_diffusion_images", recursive=True)
print(dataset)
print(dataset.first())

In [ ]:
# presumably this embeds each image then calculates the similarity between each embedding
fob.compute_uniqueness(dataset)

In [ ]:
# now we can see that `uniqueness` has been added to the list of dataset attributes
print(dataset.first())

In [ ]:
# Sort in increasing order of uniqueness (least unique first)
dups_view = dataset.sort_by("uniqueness")

# Open view in the App
session = fo.launch_app(dataset)
session.view = dups_view

This opens the GUI within the jupyter notebook. We can see the images that have been identified as the least unique, and now we can decide if any of them are duplicates.

In this case none of the images are the same; presumably they have been identified as the 'least unique' due to having similar colours and some of the people looking similar.

However, if there were duplicates, we could tag them in the GUI, and retrieve them using `session.selected`. See this tutorial for more details: https://docs.voxel51.com/tutorials/uniqueness.html.

# Image Augmentations

I will experiment with image pre-processing techniques in this notebook as it will be easier to display the images and understand how they are transformed by various functions.

Next we'll define the image augmentations.

We already identified a difference in the original sizes of the images, and decided to resize all images to 768 by 768 pixels.

We will also apply some basic transformations such as:
- horizontal flip
- random crop
- rotation

Some more complicated transformations for which we need to experiment with parameters / intensity:
- JPEG compression: requires `quality` param
- gaussian blur: requires `kernel_size` & `sigma` params

In [ ]:
# define some image paths to test with

data_root_dir = Path("/Users/ashapatel/Documents/projects/fake-image-detection/data")
folder_path = data_root_dir / "SFHQ-T2I"
image_names = os.listdir(folder_path)
test_image_paths = [folder_path / image_name for image_name in image_names]

### Experiment with JPEG compression

In [ ]:
image_index = random.randint(0,len(test_image_paths))
test_image_path = test_image_paths[image_index]
test_image_orig = Image.open(test_image_path)

plt.figure(figsize=(10,50))
num_experiments = 10
for i, jpeg_quality in enumerate(np.linspace(1,100,num_experiments)):
    test_transform = image_transforms = transforms.Compose([
        transforms.v2.JPEG(int(jpeg_quality))
    ])
    test_image_transformed = test_transform(test_image_orig)

    plt.subplot(num_experiments, 1, i+1)
    plt.imshow(test_image_transformed)
    plt.title(f"quality={jpeg_quality}")
    plt.axis("off")

With the JPEG compression transform, I don't notice a significant decrease in quality until the quality parameter is set to around 12-23. When the quality parameter gets below 12, I notice the colours in the face changing dramatically. At qualities as high as 45 I start to observe distortion in the background of the image, especially when it is originally a fairly uniform colour.

I think the higher quality values are more representative of images likely to be found online. Some quick research gives JPEG compression values for social media platforms ranging from 75 to 85. To avoid the most dramatic and less realistic changes in colour and quality, I'll set the lower bound of the transform's quality parameter to 20. 

Given how common JPEG compression is in online images, I'll set the probability of applying the transformation to 50% (p=0.5).

### Experiment with gaussian blur

In [ ]:
kernel_size = (3, 3)

image_index = random.randint(0,len(test_image_paths))
test_image_path = test_image_paths[image_index]
test_image_orig = Image.open(test_image_path)

plt.figure(figsize=(20,50))
num_experiments = 10

plt.subplot(1,2,1)
plt.imshow(test_image_orig)

test_transform = transforms.Compose([
        transforms.v2.GaussianBlur(kernel_size=kernel_size)
    ])
test_image_transformed = test_transform(test_image_orig)
plt.subplot(1,2,2)
plt.imshow(test_image_transformed)

Given that we don't expect to see significant blur in 'real-world' images, I don't think we need to set the kernel size any larger than (7,7). Note that I tested with the default sigma value set.

Also, I will set the probability of applying blur quite low; 5% (p=0.05) seems reasonable.

### Other transformations

We then need to convert the images to tensors as this is the format required by PyTorch models. This conversion also scales the pixel values to between 0-1.

Finally, we need to normalise the pixel values in the same way that the training images were normalised. Since we are fine-tuning a pre-trained CLIP model, we can find the mean and standard deviation values used to normalise the CLIP training images by importing `CLIPProcessor` from the `transformers` library.

In [ ]:
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")

clip_mean = clip_processor.image_processor.image_mean
clip_std = clip_processor.image_processor.image_std

Note that we want to apply image augmentations to the training data, but not the validation and test data as we want this to reflect 'real-world data' as closely as possible.

However, at a later point it would be interesting to produce augmented versions of the validation and test data and see how robust the model is to transformations.

In [ ]:
target_image_size = (768, 768)
prob_jpeg_compress = 0.5
prob_blur = 0.05
prob_horizontal_flip = 0.25
prob_random_crop = 0.2

train_transforms = transforms.Compose([
    transforms.Resize(size=target_image_size),
    transforms.RandomApply([
        transforms.RandomHorizontalFlip()
    ], p=prob_horizontal_flip),
    transforms.RandomApply([
        transforms.v2.RandomResizedCrop(size=target_image_size)
    ], p=prob_random_crop),
    transforms.RandomApply([
        transforms.v2.JPEG([20,100])
    ], p=prob_jpeg_compress),
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=7)
    ], p=prob_blur),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=clip_mean,
        std=clip_std
    )
])

# we still need to resize the validation & test set images, as well as converting them to tensors and normalising
val_test_transforms = transforms.Compose([
    transforms.Resize(size=target_image_size),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=clip_mean,
        std=clip_std
    )
])

In [ ]:
def view_transformed_images(
    image_paths: list[Path],
    transform: transforms.Compose,
    num_samples=3,
    seed=20,
):
    """
    Randomly sample num_samples from the list of image paths. Load, transform and display each image.
    """
    random_indices = np.random.randint(100, size=(num_samples))
    random_image_paths = [image_paths[i] for i in random_indices]
    for image_num, image_path in enumerate(random_image_paths):
        with Image.open(image_path) as image:
            fig, ax = plt.subplots(1, 2)
            ax[0].imshow(image) 
            ax[0].set_title(f"Original \nSize: {image.size}")
            ax[0].axis("off")

            # Note: permute() will change shape of image from [C, H, W] to [H, W, C] to suit matplotlib
            transformed_image = transform(image).permute(1, 2, 0) 
            ax[1].imshow(transformed_image) 
            ax[1].set_title(f"Transformed \nSize: {transformed_image.shape}")
            ax[1].axis("off")

            fig.suptitle(f"Class: {image_path.parent.stem}", fontsize=16)

            # Display pixel values to check they're as expected
            flat_image_array = np.array(torch.flatten(transformed_image))
            max_pixel = np.max(flat_image_array)
            min_pixel = np.min(flat_image_array)
            print(f"Image {image_num}: max {max_pixel}, min {min_pixel}")

In [ ]:
view_transformed_images(test_image_paths, train_transforms)

In [ ]:
view_transformed_images(test_image_paths, val_test_transforms)

# Custom PyTorch Dataset

A recap of PyTorch dataset functionality:
- `torch.utils.data.Dataset` stores the samples and their corresponding labels
    - Each time it's called, it returns an [input, label] pair
    - Pre-processing functions can be defined / called inside this class
    - A custom Dataset class must implement three functions: __init__, __len__, and __getitem__
- `torch.utils.data.DataLoader` wraps an iterable around the Dataset to enable easy access to the samples   
    - Enables iteration through the dataset in batches
    - Provides access to built-in functions for shuffling, parallel processing etc
    - Calls the `__getitem__()` function from the Dataset class to create a batch of data

In [ ]:
LABELS_DICT = {
    "real": 0,
    "synthetic": 1
}

IMAGE_FOLDER_NAMES_TO_LABELS = {
        "ffhq_real_faces": 0,
        "AIS-4SD/StableDiffusion-3-faces-20250203-1545": 1,
        "SFHQ-T2I": 1
    }

def get_samples(data_root_dir: Path) -> list[tuple[Path, int]]:
    """Get a list of samples, where each sample consists of the path to the image and the label"""
    samples = []
    for class_label_int in list(LABELS_DICT.values()):
        folder_paths = [folder_path for folder_path, class_label in IMAGE_FOLDER_NAMES_TO_LABELS.items() if class_label==class_label_int]
        for folder_path in folder_paths:
            for image_name in os.listdir(data_root_dir / folder_path):
                if image_name.lower().endswith((".png", ".jpg")):
                    image_path = data_root_dir / folder_path / image_name
                    samples.append((image_path, class_label_int))
    return samples

samples = get_samples(data_root_dir)


In [ ]:
class FaceImageDataset:

    def __init__(self, samples: list[tuple[Path, int]], transform: transforms.Compose):
        """
        Args:
            samples: list of samples where each sample consists of the path to the image and the label
            transforms: the PyTorch transforms to apply to the images in the sample
        """
        self.samples = samples
        self.transform = transform

    def __len__(self):
        """Return the total number of samples"""
        return len(self.samples)

    def __getitem__(self, idx: int):
        """
        Get one sample.
        Returns:
            transformed_image_tensor: Image tensor output by the transform function
            label: 0 for real, 1 for synthetic
        """
        image_path, label = self.samples[idx]
        image = Image.open(image_path)
        transformed_image_tensor = self.transform(image)
        assert transformed_image_tensor.shape[0] == 3, "Unexpected number of channels; expected 3 for RGB."
        return transformed_image_tensor, label    

In [ ]:
# let's start with a train/val/test split of 0.6/0.2/0.2 so that we have just over 1000 images in the validation and test sets

random.seed(10)
random.shuffle(samples)

train_size = int(0.6 * len(samples))
val_size = (len(samples) - train_size) // 2

train_samples = samples[:train_size]
val_samples = samples[train_size:train_size + val_size]
test_samples = samples[train_size + val_size:]

train_dataset = FaceImageDataset(train_samples, train_transforms)
val_dataset = FaceImageDataset(val_samples, val_test_transforms)
test_dataset = FaceImageDataset(test_samples, val_test_transforms)

print(f"\nTrain size: {len(train_dataset)}")
print(f"Val size: {len(val_dataset)}")
print(f"Test size: {len(test_dataset)}")

In [ ]:
batch_size = 64

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
def display_dataset_sample(dataloader: DataLoader):
    """
    Get a batch of images and labels from the dataloader.
    Display the first image-label pair.
    """
    images_batch, labels_batch = next(iter(dataloader))
    print(f"Image batch shape: {images_batch.size()}")
    print(f"Label batch shape: {labels_batch.size()}")
    image = images_batch[0]
    image = image.numpy().transpose(1, 2, 0)
    label_int = labels_batch[0]
    label_str = [key for key, val in LABELS_DICT.items() if val == label_int][0]
    plt.imshow(image)
    plt.title(f"Dataset image \n Label: {label_str}")
    plt.show()

In [ ]:
display_dataset_sample(train_dataloader)

In [ ]:
display_dataset_sample(val_dataloader)

In [ ]:
display_dataset_sample(test_dataloader)

Now that we have our dataloaders, we're ready to start training!